# Intraday ES Impulse PCA Research Runthrough

This notebook is the working research runthrough for the intraday branch of ChronoSwan. It studies whether large 60-minute ES1 moves have recurring cross-asset drivers, and whether event-conditioned PCA adds anything beyond a simple conditional correlation study.

Raw Bloomberg workbooks and generated outputs are local-only. The public repo carries the method, code, and unexecuted notebook; executed data-derived reports stay in `reports/` unless data licensing allows publication.

## Literature Context

This is not a blank-slate idea. Prior work covers extreme dependence, contagion/interdependence, dynamic conditional correlation, spillover networks, and PCA concentration measures of systemic risk.

- Longin and Solnik, tail correlations: https://doi.org/10.1111/0022-1082.00340
- Forbes and Rigobon, contagion versus interdependence: https://www.nber.org/papers/w7267
- Engle, DCC: https://doi.org/10.1198/073500102288618487
- Diebold and Yilmaz, spillovers: https://doi.org/10.1016/j.ijforecast.2012.08.006
- Kritzman, Li, Page, and Rigobon, PCA absorption ratio: https://doi.org/10.2469/faj.v67.n1.5

The narrower contribution here is a point-in-time ES impulse workflow: define large moves using a shifted rolling threshold, compare pairwise conditional correlations to event-conditioned PCA, and record driver-attribution tables useful in a macro/risk conversation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from chronoswan.experiments.intraday_impulse_pca import (
    build_predictive_feature_frame,
    run_intraday_impulse_pca,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

input_path = project_root / "data" / "17sheets.xlsx"
if not input_path.exists():
    input_path = project_root / "data" / "raw" / "17sheets.xlsx"

output_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"
reports_dir.mkdir(exist_ok=True)

result = run_intraday_impulse_pca(input_path=input_path, output_dir=output_dir)

coverage = result["coverage"]
long_frame = result["long_frame"]
price_panel = result["price_panel"]
return_panel = result["return_panel"]
events = result["events"]
event_summary = result["event_summary"]
corr = result["conditional_correlations"]
pca_summary = result["pca_summary"]
pca_loadings = result["pca_loadings"]
rolling_pca = result["rolling_pca"]
predictive = result["predictive_results"]
coefficients = result["predictive_coefficients"]
predictive_features = build_predictive_feature_frame(return_panel, price_panel, events)

print(f"Loaded {coverage['ticker'].nunique()} tickers from {input_path}.")
print(f"Return grid: {return_panel.index.min()} to {return_panel.index.max()}.")

Interpretation: the workbook parses cleanly into the ChronoSwan pipeline. The raw file and derived caches stay local, but the same code path can be rerun from the notebook or CLI.

## Data Audit

The first check is whether the workbook is usable as a cross-asset intraday panel. Futures and FX have broad trading clocks; ETFs and some volatility/index series have thinner cash-session-like coverage.

In [ ]:
long_frame.head(5)

Interpretation: this is the normalized OHLCV shape used by the loader. Each sheet becomes ticker-tagged rows with asset-class metadata before any return or event engineering.

In [ ]:
coverage.sort_values("rows", ascending=False).head(21)

Interpretation: coverage differs materially by instrument clock. Exact-overlap PCA should therefore be read as the complete timestamp subset, not as proof every market traded at every ES bar.

In [ ]:
sample_return_columns = [c for c in ["ES1", "NQ1", "RTY1", "TY1", "CL1", "DXY", "UX1", "UX2"] if c in return_panel]
return_panel[sample_return_columns].dropna(how="all").head(5).round(6)

Interpretation: returns are native one-bar log returns aligned by timestamp. Missing values are expected when an instrument has no bar on the same clock as another market.

## Event Definition

A large ES impulse is defined by the absolute 60-minute ES1 log return exceeding the shifted rolling 95th percentile of the prior 20-day window. The shift matters: the current bar cannot set its own threshold.

In [ ]:
events[["es_return_1h", "abs_es_return_1h", "rolling_abs_threshold", "large_abs", "large_down", "large_up"]].dropna().head(5).round(6)

Interpretation: the event table is the core point-in-time object. It turns the boss's idea into explicit large-absolute, large-down, and large-up masks.

In [ ]:
event_summary.round(4)

Interpretation: the 95th percentile rule produces a usable but still sparse event set. Down and up impulses are roughly balanced in this 140-day local window.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_frame = events.dropna(subset=["es_return_1h"])
ax.plot(plot_frame.index, plot_frame["es_return_1h"] * 10_000, linewidth=0.8, label="ES1 60m return, bp")
ax.scatter(events.index[events["large_down"]], events.loc[events["large_down"], "es_return_1h"] * 10_000, s=18, label="large down")
ax.scatter(events.index[events["large_up"]], events.loc[events["large_up"], "es_return_1h"] * 10_000, s=18, label="large up")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("ES1 hourly impulse labels")
ax.set_ylabel("basis points")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(reports_dir / "intraday_es_impulse_labels.png", dpi=150)
plt.show()

Interpretation: the plot is a visual audit of label placement. The important check is that labels cluster on genuinely large ES bars rather than ordinary intraday noise.

## Conditional Correlation Benchmark

This is the benchmark your boss explicitly asked about: before PCA, ask whether simple correlations on significant ES moves already tell the driver story.

In [ ]:
for sample in ["threshold_ready", "large_abs", "large_down", "large_up"]:
    print(f"\n{sample}")
    display(
        corr.query("sample == @sample")
        .sort_values(["abs_corr_with_es", "n_obs"], ascending=[False, False])
        .head(12)
        .round(4)
    )

Interpretation: simple conditional correlations already recover a coherent risk-off map. NQ/RTY move with ES, while VIX futures move strongly against ES during downside impulses.

## Event-Conditioned PCA

PCA is fitted on standardized driver returns, excluding ES1. Component signs are aligned so positive loadings are assets that move with positive ES returns; negative loadings tend to move against ES.

In [ ]:
pca_summary.query("status == 'fit'").round(4)

Interpretation: large absolute moves are more factor-concentrated than ordinary threshold-ready bars. This is the strongest reason to keep PCA in the workflow.

In [ ]:
for sample in ["threshold_ready", "large_abs", "large_down", "large_up"]:
    top = (
        pca_loadings.query("sample == @sample and component == 1")
        .sort_values("abs_loading", ascending=False)
        .head(10)
        [["driver", "loading_aligned_to_es", "abs_loading"]]
        .round(4)
    )
    print(f"\n{sample}: PC1 loadings")
    display(top)

Interpretation: PC1 loadings translate the factor into a driver basket. The factor is not just equity beta; volatility, oil, rates, and FX loadings explain how the ES impulse is embedded cross-asset.

## Rolling PCA Concentration

The absorption-style statistic is the fraction of standardized driver variance explained by the first three principal components. This is a concentration measure, not a directional forecast.

In [ ]:
rolling_pca[["all_bar_absorption", "large_abs_absorption", "large_abs_rows"]].describe().round(4)

Interpretation: rolling PCA concentration is consistently higher inside large ES move windows. That supports the idea that important impulses are more structured than the full intraday stream.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rolling_pca["timestamp"], rolling_pca["all_bar_absorption"], label="all bars")
ax.plot(rolling_pca["timestamp"], rolling_pca["large_abs_absorption"], label="large ES moves")
ax.set_title("Rolling PCA concentration")
ax.set_ylabel("first 3 PCs variance share")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(reports_dir / "intraday_rolling_pca_absorption.png", dpi=150)
plt.show()

Interpretation: the visual compares background factor concentration with large-move concentration through time. Gaps occur when there are too few large events in the rolling window.

## Predictive Signal Screen

This is a deliberately modest chronological screen. It asks whether known-at-bar-close cross-asset features rank next-bar large ES moves better than the train-set base rate.

In [ ]:
predictive_features.head(5).round(6)

Interpretation: these are the model inputs before fitting. Targets are next-ES-bar labels, shifted along the ES clock rather than the combined cross-asset timestamp grid.

In [ ]:
predictive.round(4)

Interpretation: unweighted logistic improves ranking but barely improves Brier score. Balanced logistic is diagnostic only because its probabilities are not calibrated for rare-event quoting.

In [ ]:
for target in coefficients["target"].drop_duplicates():
    for model in coefficients.loc[coefficients["target"].eq(target), "model"].drop_duplicates():
        print(f"\n{target} / {model}")
        display(
            coefficients.query("target == @target and model == @model")
            .head(12)
            .round(4)
        )

Interpretation: coefficients are a screening device, not causal evidence. They identify variables worth monitoring or testing with deeper history.

## Current Interpretation

The core result is whether large-move correlations and event-conditioned PCA tell the same story. If they do, PCA is mainly a compression/communication layer. If PCA reveals stable cross-asset composition not obvious from pairwise correlations, the angle becomes more interesting.

In [ ]:
large_abs_pc1 = pca_summary.query("sample == 'large_abs' and component == 1").iloc[0]
ready_pc1 = pca_summary.query("sample == 'threshold_ready' and component == 1").iloc[0]
absorption = rolling_pca[["all_bar_absorption", "large_abs_absorption"]].describe()
best_large_down_corr = corr.query("sample == 'large_down'").sort_values("abs_corr_with_es", ascending=False).head(5)
best_abs_model = predictive.query("target == 'target_next_large_abs' and model == 'logit_unweighted'").iloc[0]

print("Research readout")
print(f"- Large absolute ES impulses: PC1 explains {large_abs_pc1['explained_variance_ratio']:.1%} of driver variance versus {ready_pc1['explained_variance_ratio']:.1%} on threshold-ready bars.")
print(f"- Large-move PC1 is aligned with ES at abs corr {large_abs_pc1['abs_corr_with_es']:.2f}.")
print(f"- Median rolling absorption: all bars {absorption.loc['50%', 'all_bar_absorption']:.1%}, large ES moves {absorption.loc['50%', 'large_abs_absorption']:.1%}.")
print("- Top large-down pairwise drivers:")
for _, row in best_large_down_corr.iterrows():
    print(f"  {row['driver']}: corr {row['corr_with_es']:.2f} over {int(row['n_obs'])} observations")
print(f"- Next-bar large-absolute logistic screen: ROC AUC {best_abs_model['roc_auc']:.2f}, AP {best_abs_model['average_precision']:.2%}, Brier {best_abs_model['brier_score']:.4f}.")

Interpretation: the strongest current finding is attribution, not prediction. Large ES impulses compress into a more concentrated cross-asset factor; deeper history is needed for formal forecasting claims.